# Data Analytics for Managers Using Python — Case-Based Assessment
**Student:** Vishonika (26PGDM093) · **Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle: olistbr/brazilian-ecommerce)

The notebook follows the lecture order: Module 1 (Lectures 1–3) → Module 2 (Lectures 4–7) → Module 3 (Lectures 8–14).
Every figure is saved to `figures/` so it can be pasted into the report.

**Five business problems**
1. Late delivery and ratings — how much does lateness lower review scores, and which states are worst?
2. Freight burden — is freight a heavier share of order value in distant states, and does it hurt ratings?
3. Freight drivers — can product weight (and distance) predict the freight charge?
5. Instalments — do customers who pay in more instalments spend more per order?
8. Order profiles — what hidden profiles sit across price, freight, weight, delivery and rating (PCA)?

## Step 0 — Setup (upload `archive.zip` in the Colab Files panel first)

In [ ]:
import os, zipfile, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('olist_orders_dataset.csv'):
    with zipfile.ZipFile('archive.zip') as z:
        z.extractall('.')
os.makedirs('figures', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
SOURCE = 'Source: Olist Brazilian E-Commerce dataset (Kaggle), own Python analysis'

def save(fig, name):
    """Add the source line and save a figure for the report."""
    fig.text(0.01, -0.02, SOURCE, fontsize=8, color='grey', ha='left', va='top')
    fig.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# MODULE 1 — Python foundations (Lectures 1–3)
## Lecture 1: variables, data types and operators

In [ ]:
dataset_name = "Olist Brazilian E-Commerce"   # str
late_threshold_days = 0                         # int: late if delivered after the estimated date
currency = "BRL"                                # str: all money values are Brazilian real
sample_rate = 0.05                              # float: used to sample points for dense charts
print(type(dataset_name).__name__, type(late_threshold_days).__name__, type(sample_rate).__name__)
print("Operators:", 120 * 0.15, 10 > 7, 17 // 5, 17 % 5)

## Lecture 2: lists, dictionaries, functions, loading data with Pandas

In [ ]:
files = {   # dictionary: short name -> file
    'orders': 'olist_orders_dataset.csv', 'items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv', 'reviews': 'olist_order_reviews_dataset.csv',
    'customers': 'olist_customers_dataset.csv', 'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv', 'geo': 'olist_geolocation_dataset.csv',
    'cat_en': 'product_category_name_translation.csv'}
order_date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                   'order_delivered_customer_date', 'order_estimated_delivery_date']   # list

tables = {name: pd.read_csv(path, parse_dates=order_date_cols if name == 'orders' else None)
          for name, path in files.items()}

def quick_profile(df, name):
    """Function I wrote: rows, columns, missing cells and duplicate rows of one table."""
    return {'table': name, 'rows': df.shape[0], 'columns': df.shape[1],
            'missing_cells': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum())}

table_profile = pd.DataFrame([quick_profile(df, n) for n, df in tables.items()])
print(table_profile.to_string(index=False))

## Lecture 3: NumPy arrays, Pandas Series and DataFrames, indexing and selection

In [ ]:
orders = tables['orders']
status = orders['order_status']                                   # Series
print(status.value_counts())
delivered_mask = status == 'delivered'                             # boolean Series
days_arr = ((orders.loc[delivered_mask, 'order_delivered_customer_date'] -
             orders.loc[delivered_mask, 'order_purchase_timestamp']).dt.total_seconds().to_numpy() / 86400)  # NumPy array
print("NumPy: mean delivery days =", round(float(np.nanmean(days_arr)), 2),
      "| 90th percentile =", round(float(np.nanpercentile(days_arr, 90)), 2))
print(orders.loc[0:2, ['order_id', 'order_status']])               # .loc label selection
print(orders.iloc[0:3, 0:3])                                       # .iloc position selection

### Build the analysis tables
* `items_full` — one row per order item (used for Problem 3 regression)
* `df_raw` — one row per order (all other problems)

In [ ]:
geo = tables['geo']
# Geolocation: average lat/lng per zip prefix, Brazil bounding box only (some points fall outside Brazil)
geo_in = geo[geo['geolocation_lat'].between(-34, 6) & geo['geolocation_lng'].between(-74, -34)]
geo_bad_points = len(geo) - len(geo_in)
geo_zip = geo_in.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean()
print("Geolocation rows outside Brazil removed:", geo_bad_points)

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km, vectorised with NumPy."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

cust = tables['customers'].join(geo_zip, on='customer_zip_code_prefix').rename(
    columns={'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'})
sell = tables['sellers'].join(geo_zip, on='seller_zip_code_prefix').rename(
    columns={'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'})

items_full = (tables['items']
              .merge(tables['products'], on='product_id', how='left')
              .merge(tables['cat_en'], on='product_category_name', how='left')
              .merge(tables['orders'][['order_id', 'customer_id']], on='order_id', how='left')
              .merge(cust[['customer_id', 'customer_state', 'cust_lat', 'cust_lng']], on='customer_id', how='left')
              .merge(sell[['seller_id', 'seller_state', 'sell_lat', 'sell_lng']], on='seller_id', how='left'))
items_full['distance_km'] = haversine_km(items_full['cust_lat'], items_full['cust_lng'],
                                         items_full['sell_lat'], items_full['sell_lng'])
items_full['product_volume_cm3'] = items_full['product_length_cm'] * items_full['product_height_cm'] * items_full['product_width_cm']
print("items_full:", items_full.shape)

items_per_order = items_full.groupby('order_id').agg(
    n_items=('order_item_id', 'count'), order_price=('price', 'sum'), order_freight=('freight_value', 'sum'),
    order_weight_g=('product_weight_g', 'sum'), avg_distance_km=('distance_km', 'mean'),
    main_category=('product_category_name_english', 'first'),
    main_category_pt=('product_category_name', 'first')).reset_index()
pay = tables['payments']
pay_per_order = pay.groupby('order_id').agg(payment_value=('payment_value', 'sum'),
    max_installments=('payment_installments', 'max'), main_payment_type=('payment_type', 'first')).reset_index()
rev = tables['reviews'].sort_values('review_answer_timestamp')
rev_dupe_orders = int(rev['order_id'].duplicated().sum())
rev_per_order = rev.drop_duplicates('order_id', keep='last')[['order_id', 'review_score']]

df_raw = (orders
          .merge(tables['customers'][['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']], on='customer_id', how='left')
          .merge(items_per_order, on='order_id', how='left')
          .merge(pay_per_order, on='order_id', how='left')
          .merge(rev_per_order, on='order_id', how='left'))
df_raw['delivery_days'] = (df_raw['order_delivered_customer_date'] - df_raw['order_purchase_timestamp']).dt.total_seconds() / 86400
df_raw['days_vs_estimate'] = (df_raw['order_delivered_customer_date'] - df_raw['order_estimated_delivery_date']).dt.total_seconds() / 86400
df_raw['freight_share'] = df_raw['order_freight'] / df_raw['order_price']
df_raw['purchase_month'] = df_raw['order_purchase_timestamp'].dt.to_period('M').astype(str)
print("df_raw:", df_raw.shape, "| one row per order:", df_raw['order_id'].is_unique,
      "| orders with >1 review:", rev_dupe_orders)

# MODULE 2 — EDA and data quality (Lectures 4–7)
## Lecture 4: EDA questions; head, info, shape, describe
EDA questions: (a) How often are orders late and where? (b) How big is freight relative to price?
(c) What drives freight? (d) Do instalments go with bigger baskets? (e) Which numeric variables move together?

In [ ]:
print(df_raw.shape)
print(df_raw.head(3).T)
df_raw.info()
num_cols = df_raw.select_dtypes('number').columns
date_cols = df_raw.select_dtypes('datetime').columns
cat_cols = [c for c in df_raw.columns if c not in num_cols and c not in date_cols]
print(f"Numeric: {len(num_cols)}, Category/text: {len(cat_cols)}, Date: {len(date_cols)}")
print(df_raw.describe().T.round(2))

## Lecture 5: missing values, duplicates, inconsistent entries

In [ ]:
missing = df_raw.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))
print("Duplicate full rows:", df_raw.duplicated().sum(),
      "| duplicate order_id:", df_raw['order_id'].duplicated().sum())
print("Status x missing delivery date:\n", df_raw[df_raw['order_delivered_customer_date'].isna()]['order_status'].value_counts())
print("Delivered but no delivery date:", ((df_raw['order_status'] == 'delivered') & df_raw['order_delivered_customer_date'].isna()).sum())

# Inconsistent entries
print("payment_type values:", pay['payment_type'].value_counts().to_dict())
print("instalments = 0:", (pay['payment_installments'] == 0).sum())
untranslated = sorted(set(tables['products']['product_category_name'].dropna()) - set(tables['cat_en']['product_category_name']))
print("Categories with no English name:", untranslated)
city_raw = tables['customers']['customer_city']
print("City labels that change after strip/lower:", (city_raw != city_raw.str.strip().str.lower()).sum())
print("Delivered before purchase (negative days):", (df_raw['delivery_days'] < 0).sum())
print("Price present but zero:", (df_raw['order_price'] == 0).sum())

In [ ]:
before = {'Rows': len(df_raw), 'Missing cells': int(df_raw.isna().sum().sum()),
          'Duplicate rows': int(df_raw.duplicated().sum()),
          'Inconsistent labels': int((df_raw['main_payment_type'] == 'not_defined').sum()
                                     + (df_raw['max_installments'] == 0).sum()
                                     + df_raw['main_category_pt'].isin(untranslated).sum())}

df = df_raw.copy()
# 1) Analyse only delivered orders with a delivery date (cancelled/unavailable orders have no delivery to judge)
n_not_delivered = int((~((df['order_status'] == 'delivered') & df['order_delivered_customer_date'].notna())).sum())
df = df[(df['order_status'] == 'delivered') & df['order_delivered_customer_date'].notna()]
# 2) Drop orders with no items (no price/freight to analyse)
n_no_items = int(df['order_price'].isna().sum()); df = df.dropna(subset=['order_price'])
# 3) Inconsistent labels: fill untranslated categories, unknown category, 0 instalments -> 1, 'not_defined' payment
fix_map = {'pc_gamer': 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_food_preparers'}
df['main_category'] = df['main_category'].fillna(df['main_category_pt'].map(fix_map)).fillna('unknown')
df.loc[df['max_installments'] == 0, 'max_installments'] = 1
df = df[df['main_payment_type'].ne('not_defined') & df['payment_value'].notna()]
# 4) Missing review score: keep the order, but rating analysis uses rows with a score
n_missing_review = int(df['review_score'].isna().sum())
# 5) Late flag only on cleaned rows (every row now has a delivery date)
df['is_late'] = df['days_vs_estimate'] > late_threshold_days
df['late_label'] = np.where(df['is_late'], 'Late', 'On time')

after = {'Rows': len(df), 'Missing cells': int(df.drop(columns=['order_delivered_carrier_date', 'order_approved_at']).isna().sum().sum()),
         'Duplicate rows': int(df.duplicated().sum()),
         'Inconsistent labels': int((df['main_payment_type'] == 'not_defined').sum() + (df['max_installments'] == 0).sum()
                                    + (df['main_category'].isna()).sum())}
quality_before_after = pd.DataFrame({'Before cleaning': before, 'After cleaning': after})
print(quality_before_after)
print(f"Removed: not delivered {n_not_delivered}, no items {n_no_items}; reviews missing (kept) {n_missing_review}")
print("Missing after cleaning:\n", df.isna().sum()[df.isna().sum() > 0])

## Lecture 6: box plots, IQR outliers, retain / remove / cap / investigate

In [ ]:
def iqr_fences(s):
    """Return Q1, Q3, lower fence, upper fence and outlier count using the 1.5 x IQR rule."""
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({'Q1': q1, 'Q3': q3, 'lower_fence': lo, 'upper_fence': hi,
                      'n_outliers': int(((s < lo) | (s > hi)).sum()), 'pct_outliers': ((s < lo) | (s > hi)).mean() * 100,
                      'max': s.max()})

outlier_vars = ['order_price', 'order_freight', 'delivery_days', 'freight_share', 'order_weight_g', 'payment_value']
outlier_table = pd.DataFrame({v: iqr_fences(df[v].dropna()) for v in outlier_vars}).T.round(2)
print(outlier_table)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, v, lab in zip(axes, ['order_price', 'order_freight', 'delivery_days', 'freight_share'],
                      ['Order price (BRL)', 'Order freight (BRL)', 'Delivery time (days)', 'Freight ÷ price (ratio)']):
    sns.boxplot(y=df[v].clip(lower=0.01), ax=ax, color='#7aa6c2', fliersize=1.5, log_scale=True)
    ax.set_ylabel(lab + ' — log scale'); ax.set_title(f"{lab}\n{int(outlier_table.loc[v, 'n_outliers']):,} IQR outliers", fontsize=10)
fig.suptitle('Figure 1. Every key money and time variable has a long right tail of high values', fontweight='bold')
plt.tight_layout(); save(fig, 'fig01_boxplots_outliers')

# Decisions
long_tail_orders = int((df['delivery_days'] > 60).sum())
df['delivery_days_capped'] = df['delivery_days'].clip(upper=outlier_table.loc['delivery_days', 'upper_fence'])
df['freight_share_capped'] = df['freight_share'].clip(upper=outlier_table.loc['freight_share', 'upper_fence'])
print("Orders taking > 60 days (investigate):", long_tail_orders)

## Lecture 7: summary and group-wise statistics; managerial observations

In [ ]:
region_map = {'AC': 'North', 'AM': 'North', 'AP': 'North', 'PA': 'North', 'RO': 'North', 'RR': 'North', 'TO': 'North',
              'AL': 'Northeast', 'BA': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast', 'PE': 'Northeast',
              'PI': 'Northeast', 'RN': 'Northeast', 'SE': 'Northeast', 'DF': 'Central-West', 'GO': 'Central-West',
              'MS': 'Central-West', 'MT': 'Central-West', 'ES': 'Southeast', 'MG': 'Southeast', 'RJ': 'Southeast',
              'SP': 'Southeast', 'PR': 'South', 'RS': 'South', 'SC': 'South'}
df['region'] = df['customer_state'].map(region_map)
region_order = ['Southeast', 'South', 'Central-West', 'Northeast', 'North']

overall = {'orders': len(df), 'late_rate_pct': df['is_late'].mean() * 100,
           'avg_review': df['review_score'].mean(), 'avg_review_late': df.loc[df.is_late, 'review_score'].mean(),
           'avg_review_ontime': df.loc[~df.is_late, 'review_score'].mean(),
           'avg_delivery_days': df['delivery_days'].mean(), 'avg_freight_share_pct': df['freight_share'].median() * 100,
           'revenue_brl': df['payment_value'].sum()}
print({k: round(v, 3) for k, v in overall.items()})

by_late = df.groupby('late_label').agg(orders=('order_id', 'count'), avg_review=('review_score', 'mean'),
    pct_1_star=('review_score', lambda s: (s == 1).mean() * 100), pct_5_star=('review_score', lambda s: (s == 5).mean() * 100),
    avg_delivery_days=('delivery_days', 'mean')).round(2)
print(by_late)

by_state = df.groupby('customer_state').agg(region=('region', 'first'), orders=('order_id', 'count'),
    late_rate_pct=('is_late', lambda s: s.mean() * 100), avg_delivery_days=('delivery_days', 'mean'),
    avg_review=('review_score', 'mean'), median_freight_share_pct=('freight_share', lambda s: s.median() * 100),
    avg_order_value=('payment_value', 'mean'), revenue_brl=('payment_value', 'sum')).round(2).sort_values('late_rate_pct', ascending=False)
print(by_state)

by_region = df.groupby('region').agg(orders=('order_id', 'count'), late_rate_pct=('is_late', lambda s: s.mean() * 100),
    avg_delivery_days=('delivery_days', 'mean'), median_freight_share_pct=('freight_share', lambda s: s.median() * 100),
    avg_freight_brl=('order_freight', 'mean'), avg_price_brl=('order_price', 'mean'), avg_review=('review_score', 'mean'),
    avg_distance_km=('avg_distance_km', 'mean')).reindex(region_order).round(2)
print(by_region)

bands = pd.cut(df['max_installments'], bins=[0, 1, 3, 6, 10, 24], labels=['1 (full)', '2-3', '4-6', '7-10', '11+'])
df['installment_band'] = bands
by_inst = df.groupby('installment_band', observed=True).agg(orders=('order_id', 'count'),
    avg_payment=('payment_value', 'mean'), median_payment=('payment_value', 'median'),
    avg_items=('n_items', 'mean'), avg_review=('review_score', 'mean')).round(2)
by_inst['share_of_orders_pct'] = (by_inst['orders'] / by_inst['orders'].sum() * 100).round(1)
by_inst['share_of_revenue_pct'] = (df.groupby('installment_band', observed=True)['payment_value'].sum() / df['payment_value'].sum() * 100).round(1)
print(by_inst)

monthly = df.groupby('purchase_month').agg(orders=('order_id', 'count'), revenue_brl=('payment_value', 'sum'),
    price_brl=('order_price', 'sum'), freight_brl=('order_freight', 'sum'),
    late_rate_pct=('is_late', lambda s: s.mean() * 100), avg_review=('review_score', 'mean')).round(2)
print(monthly)

# Freight share band vs review (Problem 2)
df['freight_share_band'] = pd.cut(df['freight_share'], [0, 0.1, 0.2, 0.3, 0.5, 100], labels=['<10%', '10-20%', '20-30%', '30-50%', '>50%'])
by_fshare = df.groupby('freight_share_band', observed=True).agg(orders=('order_id', 'count'),
    avg_review=('review_score', 'mean'), avg_price=('order_price', 'mean'), avg_items=('n_items', 'mean')).round(2)
print(by_fshare)

# MODULE 3 — Visualisation (Lectures 8–10)

In [ ]:
# Figure 2 — Histogram
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(df['delivery_days'], bins=np.arange(0, 61, 1), color='#4c78a8', edgecolor='white')
med = df['delivery_days'].median()
ax.axvline(med, color='crimson', ls='--'); ax.annotate(f'Median {med:.1f} days', (med, ax.get_ylim()[1] * 0.9), xytext=(med + 3, ax.get_ylim()[1] * 0.9), color='crimson')
ax.set(title='Figure 2. About 7 in 10 orders arrive within two weeks, but a long tail waits a month or more',
       xlabel='Delivery time from purchase (days)', ylabel='Number of orders')
save(fig, 'fig02_hist_delivery_days')

In [ ]:
# Figure 3 — Scatter (item level, sample)
it = items_full.dropna(subset=['product_weight_g', 'freight_value', 'distance_km'])
it_s = it.sample(4000, random_state=1)
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(it_s['product_weight_g'] / 1000, it_s['freight_value'], c=it_s['distance_km'], cmap='viridis', s=8, alpha=0.6)
plt.colorbar(sc, label='Seller-to-customer distance (km)')
ax.set(xlim=(0, 30), ylim=(0, 200), title='Figure 3. Freight rises with product weight, and distance adds on top',
       xlabel='Product weight (kg)', ylabel='Freight charged per item (BRL)')
save(fig, 'fig03_scatter_weight_freight')

In [ ]:
# Figure 4 — Line plot, Figure 5 — Area plot (complete months only: Jan 2017 to Aug 2018)
m = monthly.loc['2017-01':'2018-08']
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(m.index, m['orders'], marker='o', color='#4c78a8')
peak = m['orders'].idxmax()
ax.annotate(f'Peak {peak}: {m.loc[peak, "orders"]:,} orders\n(Black Friday month)', (peak, m.loc[peak, 'orders']),
            xytext=(2, m.loc[peak, 'orders'] * 0.75), arrowprops=dict(arrowstyle='->'))
ax.set(title='Figure 4. Monthly orders grew fast through 2017, peaked in November, then levelled off',
       xlabel='Purchase month', ylabel='Delivered orders'); plt.xticks(rotation=60)
save(fig, 'fig04_line_monthly_orders')

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.stackplot(m.index, m['price_brl'] / 1e3, m['freight_brl'] / 1e3, labels=['Product price', 'Freight'], colors=['#4c78a8', '#f58518'], alpha=0.85)
ax.legend(loc='upper left'); plt.xticks(rotation=60)
ax.set(title='Figure 5. Freight is a steady slice of every month\'s sales value', xlabel='Purchase month', ylabel='Value (BRL thousand)')
save(fig, 'fig05_area_price_freight')

In [ ]:
# Figure 6 — Bar plot of late rate by state
bs = by_state.sort_values('late_rate_pct', ascending=False)
fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ['#d62728' if r > overall['late_rate_pct'] * 1.5 else '#9ecae1' for r in bs['late_rate_pct']]
ax.bar(bs.index, bs['late_rate_pct'], color=colors)
ax.axhline(overall['late_rate_pct'], color='black', ls='--'); ax.text(len(bs) - 1, overall['late_rate_pct'] + 0.4, f"National {overall['late_rate_pct']:.1f}%", ha='right')
ax.set(title='Figure 6. Alagoas and Maranhão are late 2.4 to 3 times as often as the national average',
       xlabel='Customer state', ylabel='Orders delivered late (%)')
save(fig, 'fig06_bar_late_by_state')

In [ ]:
# Figure 7 — Count plot
fig, (ax0, ax) = plt.subplots(1, 2, figsize=(14, 4.5))
sns.countplot(data=df.dropna(subset=['review_score']).astype({'review_score': int}), x='review_score', hue='late_label', hue_order=['On time', 'Late'],
              palette={'On time': '#4c78a8', 'Late': '#d62728'}, ax=ax0)
ax0.set(title='Count of orders', xlabel='Review score (1–5 stars)', ylabel='Number of orders'); ax0.legend(title='')
cnt =(df.dropna(subset=['review_score']).groupby('late_label')['review_score']
       .value_counts(normalize=True).mul(100).rename('pct').reset_index())
sns.barplot(data=cnt, x='review_score', y='pct', hue='late_label', hue_order=['On time', 'Late'],
            palette={'On time': '#4c78a8', 'Late': '#d62728'}, ax=ax)
ax.set_xticklabels(['1', '2', '3', '4', '5'])
ax.set(title='Share within each group', xlabel='Review score (1–5 stars)', ylabel='Share of orders in group (%)'); ax.legend(title='')
fig.suptitle('Figure 7. Late orders flip the rating pattern: 1-star becomes the most common score', fontweight='bold')
plt.tight_layout(); save(fig, 'fig07_count_review_by_late')

In [ ]:
# Figure 8 — Box plot, Figure 9 — Violin plot
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=df.dropna(subset=['review_score']).astype({'review_score': int}), x='review_score', y='delivery_days', color='#9ecae1', fliersize=1, ax=ax)
ax.set(ylim=(0, 60), title='Figure 8. Low-rated orders took far longer to arrive', xlabel='Review score (stars)', ylabel='Delivery time (days)')
save(fig, 'fig08_box_days_by_review')

fig, ax = plt.subplots(figsize=(10, 4.8))
sns.violinplot(data=df, x='region', y='freight_share_capped', order=region_order, palette='Blues', cut=0, ax=ax)
ax.set(title='Figure 9. North and Northeast customers pay the biggest freight share of product price',
       xlabel='Customer region', ylabel='Freight ÷ product price (capped at IQR fence)')
save(fig, 'fig09_violin_freight_share_region')

In [ ]:
# Figure 10 — Strip plot and swarm plot
s = df.dropna(subset=['review_score']).astype({'review_score': int}).sample(1500, random_state=2)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.stripplot(data=s, x='review_score', y='days_vs_estimate', jitter=0.3, size=2.5, alpha=0.5, ax=axes[0], color='#4c78a8')
axes[0].axhline(0, color='crimson', ls='--'); axes[0].set(ylim=(-50, 40), xlabel='Review score (stars)', ylabel='Days vs estimate (+ = late)', title='Strip plot (1,500 orders)')
sns.swarmplot(data=s.sample(400, random_state=3), x='review_score', y='days_vs_estimate', size=2.5, ax=axes[1], color='#f58518')
axes[1].axhline(0, color='crimson', ls='--'); axes[1].set(ylim=(-50, 40), xlabel='Review score (stars)', ylabel='', title='Swarm plot (400 orders)')
fig.suptitle('Figure 10. Points above the red line (late) cluster in the 1- and 2-star columns', fontweight='bold')
plt.tight_layout(); save(fig, 'fig10_strip_swarm')

In [ ]:
# Figure 11 — Factor plot / FacetGrid (catplot)
g = sns.catplot(data=df.dropna(subset=['review_score']), x='late_label', y='review_score', col='region', col_order=region_order,
                kind='bar', order=['On time', 'Late'], palette={'On time': '#4c78a8', 'Late': '#d62728'}, height=3.4, aspect=0.75)
g.set_axis_labels('', 'Average review score'); g.set_titles('{col_name}')
g.figure.suptitle('Figure 11. Lateness costs 1.7 to 1.9 stars in every region, so it is not a regional quirk', y=1.05, fontweight='bold')
save(g.figure, 'fig11_facet_review_late_region')

In [ ]:
# Figure 12 — Distribution (KDE)
fig, ax = plt.subplots(figsize=(9, 4.5))
for band, col in zip(['1 (full)', '2-3', '4-6', '7-10'], ['#9ecae1', '#6baed6', '#3182bd', '#08519c']):
    sns.kdeplot(np.log10(df.loc[df['installment_band'] == band, 'payment_value']), label=band, ax=ax, color=col, lw=2)
ax.set_xticks([1, 1.5, 2, 2.5, 3, 3.5]); ax.set_xticklabels(['10', '32', '100', '316', '1,000', '3,162'])
ax.set(title='Figure 12. Each step up in instalments shifts the order-value curve to the right',
       xlabel='Order payment value (BRL, log scale)', ylabel='Density'); ax.legend(title='Instalments')
save(fig, 'fig12_kde_payment_by_installments')

In [ ]:
# Figure 13 — Correlation heatmap (Lecture 12)
corr_vars = ['order_price', 'order_freight', 'order_weight_g', 'avg_distance_km', 'n_items', 'delivery_days',
             'days_vs_estimate', 'review_score', 'max_installments', 'payment_value', 'freight_share']
corr = df[corr_vars].corr().round(2)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, square=True, cbar_kws={'label': 'Pearson r'})
ax.set_title('Figure 13. Price, payment and freight move together; delay is the main thing that moves against rating', fontweight='bold')
save(fig, 'fig13_heatmap_correlation')

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().rename('r').reset_index()
pairs['abs_r'] = pairs['r'].abs()
print(pairs.sort_values('abs_r', ascending=False).head(10).to_string(index=False))
print("Corr days_vs_estimate vs review:", corr.loc['days_vs_estimate','review_score'])
print("Share of orders delivered within 14 days:", round((df['delivery_days'] <= 14).mean()*100, 1))
print("Rating gap (on time - late) by region:", (df[~df.is_late].groupby('region')['review_score'].mean() - df[df.is_late].groupby('region')['review_score'].mean()).round(2).to_dict())
print("Freight/price ratio by month (min, max):", round((m['freight_brl']/m['price_brl']).min(),3), round((m['freight_brl']/m['price_brl']).max(),3))
print("Spearman instalments vs payment:", round(df['max_installments'].corr(df['payment_value'], method='spearman'), 3))

In [ ]:
# Figure 14 — Pair plot
pp = df[['order_price', 'order_freight', 'delivery_days', 'review_score', 'late_label']].dropna().sample(800, random_state=4)
pp = pp[(pp['order_price'] < 1000) & (pp['order_freight'] < 100) & (pp['delivery_days'] < 60)]
g = sns.pairplot(pp, hue='late_label', hue_order=['On time', 'Late'], palette={'On time': '#4c78a8', 'Late': '#d62728'},
                 plot_kws={'s': 10, 'alpha': 0.6}, height=2.1)
g.figure.suptitle('Figure 14. Late orders separate clearly on delivery time and rating, not on price', y=1.02, fontweight='bold')
save(g.figure, 'fig14_pairplot')

# Statistics and relationships (Lectures 11–14)
## Lecture 11: mean, median, mode, variance, standard deviation, quartiles

In [ ]:
def describe_stats(s):
    s = s.dropna()
    return pd.Series({'mean': s.mean(), 'median': s.median(), 'mode': s.round(0).mode().iloc[0],
                      'variance': s.var(), 'std_dev': s.std(), 'Q1': s.quantile(0.25), 'Q3': s.quantile(0.75)})
stats_table = pd.DataFrame({v: describe_stats(df[v]) for v in ['payment_value', 'order_freight', 'delivery_days', 'review_score', 'max_installments']}).T.round(2)
print(stats_table)
print("Review score mode (exact):", df['review_score'].mode().iloc[0], "| Payment type mode:", df['main_payment_type'].mode().iloc[0])

## Lecture 12: correlation vs causation check
Lateness and rating move together. Distance could drive both (far customers get late orders *and* may rate differently),
so compare the rating gap inside one distance band.

In [ ]:
df['distance_band'] = pd.qcut(df['avg_distance_km'], 4, labels=['Nearest 25%', 'Q2', 'Q3', 'Farthest 25%'])
gap_by_dist = df.pivot_table(index='distance_band', columns='late_label', values='review_score', aggfunc='mean', observed=True).round(2)
gap_by_dist['gap'] = (gap_by_dist['On time'] - gap_by_dist['Late']).round(2)
print(gap_by_dist)

## Lecture 13: simple linear regression — freight per item on product weight (Problem 3)

In [ ]:
reg = items_full.dropna(subset=['product_weight_g', 'freight_value', 'distance_km']).copy()
reg['weight_kg'] = reg['product_weight_g'] / 1000
X, y = reg[['weight_kg']].to_numpy(), reg['freight_value'].to_numpy()
lr = LinearRegression().fit(X, y)
r2 = lr.score(X, y); b0, b1 = lr.intercept_, lr.coef_[0]
resid = y - lr.predict(X)
print(f"Freight = {b0:.2f} + {b1:.2f} x weight_kg | R² = {r2:.3f} | n = {len(reg):,}")
print(f"Residual mean {resid.mean():.3f}, residual SD {resid.std():.2f}, RMSE {np.sqrt((resid**2).mean()):.2f}")
pred_5kg = lr.predict([[5]])[0]; print(f"Prediction for a 5 kg item: BRL {pred_5kg:.2f}")

# Extension: add distance (multiple regression) to see how much fit improves
X2 = reg[['weight_kg', 'distance_km']].to_numpy()
lr2 = LinearRegression().fit(X2, y); r2_2 = lr2.score(X2, y)
print(f"Freight = {lr2.intercept_:.2f} + {lr2.coef_[0]:.2f} x weight_kg + {lr2.coef_[1]:.4f} x distance_km | R² = {r2_2:.3f}")
print(f"Each extra 1,000 km adds BRL {lr2.coef_[1]*1000:.2f}")
pred_5kg_far = lr2.predict([[5, 2000]])[0]; pred_5kg_near = lr2.predict([[5, 200]])[0]
print(f"5 kg item at 200 km: BRL {pred_5kg_near:.2f}; at 2,000 km: BRL {pred_5kg_far:.2f}")
# Overcharged items: actual freight well above the model
reg['resid2'] = y - lr2.predict(X2)
over = reg[reg['resid2'] > 2 * reg['resid2'].std()]
print(f"Items charged > 2 SD above model: {len(over):,} ({len(over)/len(reg)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rs = reg.sample(5000, random_state=5)
axes[0].scatter(rs['weight_kg'], rs['freight_value'], s=5, alpha=0.3, color='#4c78a8')
xs = np.linspace(0, 30, 50); axes[0].plot(xs, b0 + b1 * xs, color='crimson', lw=2, label=f'Freight = {b0:.2f} + {b1:.2f}×kg (R² = {r2:.2f})')
axes[0].set(xlim=(0, 30), ylim=(0, 250), xlabel='Product weight (kg)', ylabel='Freight per item (BRL)', title='Fitted line'); axes[0].legend()
axes[1].scatter(lr.predict(rs[['weight_kg']].to_numpy()), rs['freight_value'] - lr.predict(rs[['weight_kg']].to_numpy()), s=5, alpha=0.3, color='#f58518')
axes[1].axhline(0, color='black'); axes[1].set(ylim=(-100, 150), xlabel='Predicted freight (BRL)', ylabel='Residual (actual − predicted, BRL)', title='Residuals')
fig.suptitle(f'Figure 15. Every extra kilogram adds about BRL {b1:.2f} of freight, but weight alone explains only {r2*100:.0f}% of it', fontweight='bold')
plt.tight_layout(); save(fig, 'fig15_regression_residuals')

In [ ]:
# Figure 16 — Mesh grid, contour and quiver: freight surface from the two-variable model
W, D = np.meshgrid(np.linspace(0, 30, 40), np.linspace(0, 3000, 40))
F = lr2.intercept_ + lr2.coef_[0] * W + lr2.coef_[1] * D
fig, ax = plt.subplots(figsize=(9, 5.5))
cs = ax.contourf(W, D, F, levels=12, cmap='YlOrRd'); plt.colorbar(cs, label='Predicted freight (BRL)')
ax.contour(W, D, F, levels=12, colors='k', linewidths=0.3)
Wq, Dq = np.meshgrid(np.linspace(2, 28, 8), np.linspace(200, 2800, 8))
# Gradient per grid step (each axis split into 40 steps), so arrow direction compares the two drivers fairly
dF_dD, dF_dW = np.gradient(F)
ax.quiver(W[::5, ::5], D[::5, ::5], dF_dW[::5, ::5], dF_dD[::5, ::5], angles='uv', color='navy', alpha=0.8)
print(f"Freight rise across plotted range: weight 0-30 kg = BRL {lr2.coef_[0]*30:.1f}; distance 0-3,000 km = BRL {lr2.coef_[1]*3000:.1f}")
ax.set(xlabel='Product weight (kg)', ylabel='Seller-to-customer distance (km)',
       title='Figure 16. Over typical ranges, weight pushes freight up more than twice as much as distance')
save(fig, 'fig16_meshgrid_contour_quiver')

## Lecture 14: PCA — order profiles (Problem 8)

In [ ]:
pca_vars = ['order_price', 'order_freight', 'order_weight_g', 'avg_distance_km', 'n_items',
            'delivery_days', 'days_vs_estimate', 'review_score', 'max_installments']
P = df[pca_vars].dropna()
Z = StandardScaler().fit_transform(P)
pca = PCA().fit(Z)
evr = pca.explained_variance_ratio_
loadings = pd.DataFrame(pca.components_.T, index=pca_vars, columns=[f'PC{i+1}' for i in range(len(pca_vars))]).round(2)
pca_table = pd.DataFrame({'variance_explained_pct': (evr * 100).round(1), 'cumulative_pct': (evr.cumsum() * 100).round(1),
                          'eigenvalue': pca.explained_variance_.round(2)}, index=loadings.columns)
print(pca_table); print(loadings[['PC1', 'PC2', 'PC3']])
n_keep = int((pca.explained_variance_ > 1).sum()); print("Components with eigenvalue > 1:", n_keep)

scores = pca.transform(Z)[:, :3]
P_sc = P.assign(PC1=scores[:, 0], PC2=scores[:, 1], PC3=scores[:, 2])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(pca_table.index, pca_table['variance_explained_pct'], color='#4c78a8')
axes[0].plot(pca_table.index, pca_table['cumulative_pct'], color='crimson', marker='o', label='Cumulative %')
for i, v in enumerate(pca_table['cumulative_pct'][:3]): axes[0].annotate(f'{v}%', (i, v + 2), ha='center', color='crimson')
axes[0].set(ylabel='Variance explained (%)', xlabel='Principal component', title='Scree plot'); axes[0].legend()
ps = P_sc.sample(3000, random_state=6)
sc = axes[1].scatter(ps['PC1'], ps['PC2'], c=ps['review_score'], cmap='RdYlGn', s=6, alpha=0.7)
plt.colorbar(sc, ax=axes[1], label='Review score')
for v in pca_vars:
    axes[1].arrow(0, 0, loadings.loc[v, 'PC1'] * 4, loadings.loc[v, 'PC2'] * 4, color='black', head_width=0.1)
    axes[1].text(loadings.loc[v, 'PC1'] * 4.4, loadings.loc[v, 'PC2'] * 4.4, v, fontsize=8)
axes[1].set(xlim=(-4, 8), ylim=(-5, 6), xlabel='PC1', ylabel='PC2', title='Biplot (3,000 orders)')
fig.suptitle(f'Figure 17. Three profiles — basket size, delivery experience, distance — explain {pca_table["cumulative_pct"].iloc[2]}% of order variation', fontweight='bold')
plt.tight_layout(); save(fig, 'fig17_pca_scree_biplot')

In [ ]:
# Figure 18 — Multi-panel (subplots) with annotations: Problem 1 dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(m.index, m['late_rate_pct'], marker='o', color='#d62728'); axes[0, 0].tick_params(axis='x', rotation=70)
worst_m = m['late_rate_pct'].idxmax()
axes[0, 0].annotate(f'{worst_m}: {m.loc[worst_m, "late_rate_pct"]:.1f}% late', (worst_m, m.loc[worst_m, 'late_rate_pct']),
                    xytext=(1, m.loc[worst_m, 'late_rate_pct'] * 0.95), arrowprops=dict(arrowstyle='->'))
axes[0, 0].set(title='A. Late rate by month', ylabel='Late orders (%)')
axes[0, 1].plot(m.index, m['avg_review'], marker='o', color='#4c78a8'); axes[0, 1].tick_params(axis='x', rotation=70)
axes[0, 1].set(title='B. Average rating by month (dips when lateness spikes)', ylabel='Average review score')
by_late['avg_review'].reindex(['On time', 'Late']).plot.bar(ax=axes[1, 0], color=['#4c78a8', '#d62728'], rot=0)
for i, v in enumerate(by_late['avg_review'].reindex(['On time', 'Late'])): axes[1, 0].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')
axes[1, 0].set(title='C. Average rating, on time vs late', ylabel='Average review score', xlabel='', ylim=(0, 5))
br = by_region.reset_index()
axes[1, 1].scatter(br['avg_distance_km'], br['late_rate_pct'], s=br['orders'] / 100, color='#f58518', alpha=0.7)
for _, r_ in br.iterrows(): axes[1, 1].annotate(r_['region'], (r_['avg_distance_km'], r_['late_rate_pct']), xytext=(5, 5), textcoords='offset points')
axes[1, 1].set(title='D. The Northeast is late most often, more than the even farther North (bubble = orders)', xlabel='Average seller distance (km)', ylabel='Late orders (%)')
fig.suptitle('Figure 18. Lateness spikes in specific months and in the Northeast, and ratings fall with it', fontweight='bold', fontsize=14)
plt.tight_layout(); save(fig, 'fig18_subplots_problem1')

In [ ]:
# Figure 19 — Map: late rate by state at the state's average customer location
state_xy = cust.groupby('customer_state')[['cust_lat', 'cust_lng']].mean()
mp = by_state.join(state_xy)
fig, ax = plt.subplots(figsize=(8, 8))
bg = geo_zip.sample(min(15000, len(geo_zip)), random_state=7)
ax.scatter(bg['geolocation_lng'], bg['geolocation_lat'], s=0.3, color='lightgrey')
sc = ax.scatter(mp['cust_lng'], mp['cust_lat'], s=mp['orders'] / 40 + 30, c=mp['late_rate_pct'], cmap='Reds', edgecolor='k', vmin=0)
for st, r_ in mp.iterrows(): ax.annotate(st, (r_['cust_lng'], r_['cust_lat']), fontsize=8, ha='center', va='center')
plt.colorbar(sc, label='Orders delivered late (%)', shrink=0.7)
ax.set(xlabel='Longitude', ylabel='Latitude', title='Figure 19. Late-delivery hot spots: the Northeast coast and Rio de Janeiro')
ax.set_aspect('equal'); save(fig, 'fig19_map_late_rate')

## Business-impact calculations used in Sections 6–7

In [ ]:
# P1: if the 10 worst states fell to the national late rate
worst10 = by_state.sort_values('late_rate_pct', ascending=False).head(10)
late_now = (worst10['late_rate_pct'] / 100 * worst10['orders']).sum()
late_target = (overall['late_rate_pct'] / 100 * worst10['orders']).sum()
avoidable_late = late_now - late_target
review_gain_total = avoidable_late * (overall['avg_review_ontime'] - overall['avg_review_late'])
print(f"Worst-10 states: {int(worst10['orders'].sum()):,} orders, {late_now:.0f} late now; at national rate {late_target:.0f}; avoidable {avoidable_late:.0f}")
print("Worst 10 states:", list(worst10.index))
# Share of late orders that came from estimate vs real slowness: what if estimate were +3 days?
late_within_3 = ((df['days_vs_estimate'] > 0) & (df['days_vs_estimate'] <= 3)).sum()
print(f"Late by 3 days or less: {late_within_3:,} of {int(df['is_late'].sum()):,} late orders ({late_within_3/df['is_late'].sum()*100:.1f}%)")
print(f"Orders delivered early: {(df['days_vs_estimate'] < 0).mean()*100:.1f}% | median days early {(-df.loc[~df.is_late,'days_vs_estimate']).median():.1f}")
# P2: freight above 30% of price
print(f"Orders with freight > 30% of price: {(df['freight_share'] > 0.3).mean()*100:.1f}%")
print("North+Northeast share of orders:", round(df['region'].isin(['North', 'Northeast']).mean() * 100, 1),
      "% | of revenue:", round(df.loc[df['region'].isin(['North', 'Northeast']), 'payment_value'].sum() / df['payment_value'].sum() * 100, 1), '%')
# P5: credit-card orders
cc = df[df['main_payment_type'] == 'credit_card']
print(f"Credit card share of orders: {len(cc)/len(df)*100:.1f}% | of these paying >1 instalment: {(cc['max_installments'] > 1).mean()*100:.1f}%")
print("Corr instalments vs payment (credit card only):", round(cc['max_installments'].corr(cc['payment_value']), 3))
# P8: profile groups on PC1
P_sc['basket_profile'] = pd.qcut(P_sc['PC1'], [0, .5, .9, 1], labels=['Typical', 'Large', 'Top 10% heavy/large'])
print(P_sc.groupby('basket_profile', observed=True)[['order_price', 'order_freight', 'order_weight_g', 'max_installments', 'review_score']].mean().round(2))
P_sc['delay_profile'] = pd.qcut(P_sc['PC2'], [0, .9, 1], labels=['Normal', 'Top 10% PC2'])
print(P_sc.groupby('delay_profile', observed=True)[['delivery_days', 'days_vs_estimate', 'review_score', 'avg_distance_km']].mean().round(2))

## Export for the Excel workbook

In [ ]:
summary_tables = {'Table_Profile': table_profile, 'Quality_Before_After': quality_before_after.reset_index(names='Measure'),
    'Outliers_IQR': outlier_table.reset_index(names='variable'), 'By_Late': by_late.reset_index(), 'By_State': by_state.reset_index(),
    'By_Region': by_region.reset_index(), 'By_Instalment': by_inst.reset_index(), 'By_Freight_Share': by_fshare.reset_index(),
    'Monthly': monthly.reset_index(), 'Stats_L11': stats_table.reset_index(names='variable'), 'Correlation': corr.reset_index(names='variable'),
    'Top_Corr_Pairs': pairs.sort_values('abs_r', ascending=False).head(10), 'Gap_By_Distance': gap_by_dist.reset_index(),
    'PCA_Variance': pca_table.reset_index(names='component'), 'PCA_Loadings': loadings.reset_index(names='variable')}

## Save the Excel workbook (raw data, cleaned data, summary tables, chart data)

In [ ]:
with pd.ExcelWriter('26PGDM093_Vishonika_CaseAssessment.xlsx') as xw:
    tables['orders'].to_excel(xw, sheet_name='Raw_Orders', index=False)
    df.drop(columns=['main_category_pt']).to_excel(xw, sheet_name='Cleaned_Orders', index=False)
    for name, t in summary_tables.items():
        t.to_excel(xw, sheet_name=name[:31], index=False)
print('Workbook saved')